# E15：两组比较与统计结果解释

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：验证性与综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 识别独立两组比较中的观测单位、分组变量和结果变量。
2. 使用描述统计和图形检查两组样本。
3. 调用 Welch 独立样本 t 检验并读取结果对象。
4. 按固定方向报告均值差、95% 置信区间和 p 值。
5. 区分样本差异、统计证据、实际意义和因果解释。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、统计结果、图形和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的官方分析视图与检查点。
4. 提交前重启内核并按顺序运行全部单元，核对变量角色、统计口径和结果方向。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的路径和输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，后续结果才会更新。
- 统计结果异常时，先检查分组方向、变量名、缺失值和数据类型。
- 如果变量状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 输入、变量角色和适用条件：10 分钟
- 分组描述统计：10 分钟
- 图形检查：15 分钟
- Welch t 检验：15 分钟
- 均值差与置信区间：15 分钟
- 结果表和导出：10 分钟
- 规范解释、核验与提交：15 分钟

## 主线输入：继续 E12 成果，官方分析视图兜底

课程准备代码优先读取 E12 个人成果，并选择第六章所需字段；个人成果不存在时使用官方 `student_analysis.csv`。个人成果一旦被选中但核验失败，程序会停止。本实验固定比较 `feedback` 与 `regular` 两组，并把差异方向定义为 `feedback - regular`。

### 研究设计背景

这套模拟数据中的 `support_plan` 来自既有教学安排，并非随机分配。学生分析表中每名学生只出现一次，不涉及同一学生的重复测量；本实验在分析中将不同学生视为相互独立的观测。是否随机分配、观测是否独立等信息来自研究设计，不能只根据 CSV 中的数值判断。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

# 有可靠的 E12 个人成果时保持 True；需要从官方分析视图开始时改为 False。
USE_STUDENT_E12_OUTPUT = True

chapter6_dir = Path(".").resolve()
notebook_names = ["e15.ipynb", "e15_inclass.ipynb"]
if not any((chapter6_dir / name).exists() for name in notebook_names):
    chapter6_dir = Path("lab/applied-stat-64/chapter6").resolve()

shared_dir = chapter6_dir.parents[1] / "shared"
student_e12_path = (
    chapter6_dir.parent
    / "chapter4"
    / "e12_output"
    / "e12_analysis_ready.csv"
)
official_path = shared_dir / "analysis" / "student_analysis.csv"
output_dir = chapter6_dir / "e15_output"

analysis_columns = [
    "student_id",
    "support_plan",
    "baseline_score",
    "study_hours",
    "attendance_rate",
    "final_score",
]
plan_order = ["regular", "feedback"]

if USE_STUDENT_E12_OUTPUT and student_e12_path.exists():
    source_path = student_e12_path
    raw_analysis = pd.read_csv(source_path)
    missing_columns = [
        column
        for column in analysis_columns
        if column not in raw_analysis.columns
    ]
    if missing_columns:
        raise ValueError(
            f"E12 个人成果缺少字段：{missing_columns}。"
            "请修正个人成果，或把 USE_STUDENT_E12_OUTPUT 改为 False。"
        )
    analysis_df = raw_analysis[analysis_columns].copy()
    print("输入来源：E12 个人成果")
else:
    if USE_STUDENT_E12_OUTPUT:
        print("未找到 E12 个人成果，改用官方分析视图。")
    source_path = official_path
    analysis_df = pd.read_csv(source_path)
    print("输入来源：官方分析视图")

checks = {
    "字段及顺序正确": analysis_df.columns.tolist() == analysis_columns,
    "记录数为 160": len(analysis_df) == 160,
    "student_id 唯一": analysis_df["student_id"].is_unique,
    "正式字段无缺失": not analysis_df.isna().any().any(),
    "支持方案取值正确": (
        set(analysis_df["support_plan"]) == set(plan_order)
    ),
    "每组 80 名学生": (
        analysis_df["support_plan"].value_counts().eq(80).all()
    ),
}
for check_name, passed in checks.items():
    print(check_name, "：", passed)
if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise ValueError("输入核验未通过：" + "、".join(failed))

analysis_df["support_plan"] = pd.Categorical(
    analysis_df["support_plan"],
    categories=plan_order,
    ordered=True,
)
print("输入文件：", source_path)
print("数据形状：", analysis_df.shape)

## 任务 1：识别研究问题、变量角色和检查边界

研究问题：当前模拟样本中，`feedback` 与 `regular` 两种支持方案的期末成绩是否存在差异？

运行代码后，用 Markdown 说明：

1. 一行代表什么，两个组是否由不同学生构成；
2. 哪一列是分组变量，哪一列是数值结果变量；
3. 哪些条件可以从数据中检查，哪些必须由研究设计说明；
4. 为什么不使用一串前置检验机械选择方法。

In [ ]:
print(analysis_df.dtypes)
print("\n分组人数：")
print(
    analysis_df["support_plan"]
    .value_counts()
    .reindex(...)
)
print("\n结果变量摘要：")
print(analysis_df[...].describe().round(2))
print("\n缺失值：")
print(
    analysis_df[
        [...]
    ].isna().sum()
)

## 任务 2：先完成分组描述统计

按 `support_plan` 分组，对 `final_score` 计算样本量、均值、中位数和标准差，并按 `regular`、`feedback` 的顺序展示。先回答“当前样本中看到了什么”，再进入统计检验。

In [ ]:
group_summary = (
    analysis_df.groupby(
        ...,
        observed=True,
    )["final_score"]
    .agg([...])
    .reindex(plan_order)
    .round(2)
)
group_summary

## 任务 3：使用箱线图和个体点检查两组分布

在同一个 `Axes` 上叠加箱线图和个体点。箱线图使用 `showfliers=False`，让所有真实观测统一由 `stripplot()` 展示，避免可能离群的记录被重复绘制。根据图形观察组间重叠、离散程度和可能的极端记录，但不要把图形差异直接写成总体或因果结论。

In [ ]:
group_fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=analysis_df,
    x=...,
    y=...,
    order=plan_order,
    color="#D9E6F2",
    width=0.55,
    showfliers=False,
    ax=ax,
)
sns.stripplot(
    data=analysis_df,
    x=...,
    y=...,
    order=plan_order,
    color="#333333",
    alpha=0.40,
    size=3,
    ax=ax,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
group_fig.tight_layout()
plt.show()

## 任务 4：完成 Welch t 检验并固定差异方向

显式构造 `feedback_scores` 和 `regular_scores`，再调用：

```python
stats.ttest_ind(
    feedback_scores,
    regular_scores,
    equal_var=False,
)
```

均值差、置信区间和 t 统计量都必须保持 `feedback - regular` 的方向。按照“均值差 → 置信区间 → p 值”的顺序读取结果。

In [ ]:
feedback_scores = analysis_df.loc[
    analysis_df["support_plan"] == ...,
    ...,
]
regular_scores = analysis_df.loc[
    analysis_df["support_plan"] == ...,
    ...,
]

test_result = stats.ttest_ind(
    ...,
    ...,
    equal_var=...,
)
mean_difference = ... - ...
confidence_interval = test_result.confidence_interval(
    confidence_level=...
)

print("feedback 组均值：", round(feedback_scores.mean(), 3))
print("regular 组均值：", round(regular_scores.mean(), 3))
print("均值差 feedback - regular：", round(mean_difference, 3))
print(
    "95% 置信区间：",
    (
        round(confidence_interval.low, 3),
        round(confidence_interval.high, 3),
    ),
)
print("t 统计量：", round(test_result.statistic, 3))
print("p 值：", round(test_result.pvalue, 4))

## 任务 5：整理并导出结构化比较结果

把 E16 需要继续使用的结果整理为一行 DataFrame。字段名称和组序由课程规定，不自行交换；同时导出分组图和 `e15_group_comparison.csv`。

In [ ]:
comparison_result = pd.DataFrame(
    [
        {
            "outcome": "final_score",
            "group_1": "feedback",
            "group_2": "regular",
            "n_group_1": ...,
            "n_group_2": ...,
            "mean_group_1": ...,
            "mean_group_2": ...,
            "mean_difference": ...,
            "ci_low": ...,
            "ci_high": ...,
            "t_statistic": ...,
            "p_value": ...,
        }
    ]
).round(6)

output_dir.mkdir(parents=True, exist_ok=True)
result_path = output_dir / "e15_group_comparison.csv"
figure_path = output_dir / "e15_group_distribution.png"
comparison_result.to_csv(result_path, index=False)
group_fig.savefig(figure_path, dpi=150, bbox_inches="tight")

print(comparison_result.T)
print("结果文件：", result_path)
print("图形文件：", figure_path)

## 任务 6：完成统计摘要并核验 AI 解释

按以下顺序写一段结果：

1. 两组样本量和平均成绩；
2. `feedback - regular` 的均值差；
3. 95% 置信区间；
4. Welch t 检验的 p 值；
5. 当前数据支持的统计结论；
6. 实际意义与因果解释边界。

某 AI 工具声称：“p 值为 0.007，说明即时反馈有 99.3% 的概率提高所有学生成绩，而且每名学生都会提高 4.20 分。”请指出至少三处错误并改写。

### 统计摘要与 AI 核验

...

AI 表述的主要问题：

1. ...
2. ...
3. ...

改写：...

## 任务 7：最终核验与实验小结

运行检查代码后，用 Markdown 补充：

1. 为什么结果方向必须始终保持 `feedback - regular`？
2. 为什么独立性不能只从 CSV 数值中判断？
3. 本实验是否使用了帮助文档、代码补全或 AI 工具，你如何核验其建议？

In [ ]:
final_checks = {
    "结果文件已导出": result_path.exists(),
    "图形文件已导出": figure_path.exists(),
    "组序正确": (
        comparison_result.loc[0, "group_1"] == "feedback"
        and comparison_result.loc[0, "group_2"] == "regular"
    ),
    "每组样本量为 80": (
        comparison_result.loc[0, "n_group_1"] == 80
        and comparison_result.loc[0, "n_group_2"] == 80
    ),
    "均值差方向一致": (
        np.sign(comparison_result.loc[0, "mean_difference"])
        == np.sign(comparison_result.loc[0, "t_statistic"])
    ),
    "置信区间顺序正确": (
        comparison_result.loc[0, "ci_low"]
        < comparison_result.loc[0, "ci_high"]
    ),
    "p 值范围正确": (
        0 <= comparison_result.loc[0, "p_value"] <= 1
    ),
}
for check_name, passed in final_checks.items():
    print(check_name, "：", passed)
if not all(final_checks.values()):
    raise ValueError("最终核验未通过。")

print(
    "\n实验小结：结果方向应保持一致；独立性来自研究设计，"
    "不能只由数值表判断；工具建议必须通过代码运行和结果口径核验。"
)